In [ ]:
import os, sys, warnings
from pathlib import Path
_r = Path.cwd()
while not (_r / 'requirements.txt').exists() and _r != _r.parent:
    _r = _r.parent
os.chdir(_r); sys.path.insert(0, str(_r / 'src')); warnings.filterwarnings('ignore')

# Degraded vehicles — forecast to 80% EOL, per-model warranty

Oldest vehicles with a genuine arc (≥`START_MIN`% → ≤`EOL_REACH`%). Forecast (upgraded `src/model.py`
rate model) is drawn **only until 80% SoH**. Warranty per model from the spec sheet
(`src/config.py`): Treo 3 yr / Zor Grand 5 yr, or 120k km — whichever first (km projected from odometer).

In [ ]:
import numpy as np, pandas as pd, pyarrow as pa, pyarrow.dataset as ds
import matplotlib; import matplotlib.pyplot as plt
from config import warranty_for
import model

START_MIN, EOL_REACH, MIN_MONTHS, EOL = 93.0, 88.0, 8, 80.0
m = pd.read_parquet('data/mahindra/features/feature_table.parquet').sort_values(['vin','month'])
_d0 = ds.dataset('data/mahindra/intellicar', format='parquet')
_sch = pa.schema([(pa.field('model', pa.string()) if f.name=='model' else f) for f in _d0.schema])
_ap = ds.dataset('data/mahindra/intellicar', format='parquet', schema=_sch).to_table(columns=['vin','model']).group_by('vin').aggregate([('model','min')]).to_pandas()
vin_model = dict(zip(_ap['vin'], _ap['model_min']))

models = model.train_quantiles(model.build_transitions(m))
sel=[]
for vin,g in m.groupby('vin'):
    if len(g)<MIN_MONTHS: continue
    if g['soh'].max()>=START_MIN and g['soh'].min()<=EOL_REACH:
        sel.append((g['age_months'].max(),vin,g))
sel.sort(key=lambda z:-z[0])
print('models in cohort:', pd.Series(vin_model).value_counts().to_dict())
print(f'{len(sel)} oldest vehicles with a {START_MIN:.0f}->{EOL_REACH:.0f}% arc')

In [ ]:
def warranty_end(g, start):
    yrs, kmlim = warranty_for('mahindra', vin_model.get(g['vin'].iloc[0], 'Treo'))
    age = g['age_months'].max(); t_date = start + pd.DateOffset(years=int(yrs))
    od = g.dropna(subset=['odo_max']).sort_values('month')
    if len(od) >= 2:
        span = max((od['month'].iloc[-1]-od['month'].iloc[0]).days/30.4, 1.0)
        odo_now = od['odo_max'].iloc[-1]; km_pm = max(od['odo_max'].iloc[-1]-od['odo_max'].iloc[0],0)/span
        last_date = od['month'].iloc[-1]
    else:
        odo_now, km_pm, last_date = np.nan, np.nan, start + pd.to_timedelta(age*30.4375, unit='D')
    km_months = min(max(kmlim-odo_now,0)/km_pm, 1200.0) if (np.isfinite(odo_now) and np.isfinite(km_pm) and km_pm>0) else 1200.0
    km_date = last_date + pd.to_timedelta(km_months*30.4375, unit='D')
    return (km_date, f'{kmlim//1000}k km') if km_date <= t_date else (t_date, f'{int(yrs)} yr')

n=len(sel); ncol=min(4,n) if n else 1; nrow=int(np.ceil(n/ncol)) if n else 1
fig,axes=plt.subplots(nrow,ncol,figsize=(4.6*ncol,3.3*nrow),squeeze=False)
for ax in axes.flat: ax.axis('off')
MAXH=120
for i,(age,vin,g) in enumerate(sel):
    ax=axes[i//ncol][i%ncol]; ax.axis('on'); g=g.sort_values('month')
    start=g['month'].min(); t=g['age_months'].to_numpy(); s=g['soh'].to_numpy()
    we,wlab=warranty_end(g,start)
    sim=model.simulate(g,models,MAXH); med=sim['q50'].to_numpy(); cross=np.where(med<=EOL)[0]
    k=int(cross[0]+1) if len(cross) else len(sim)
    sim=sim.iloc[:k]; fa=t[-1]+np.arange(1,k+1)
    d0=start+pd.to_timedelta(t*30.4375,unit='D'); df_=start+pd.to_timedelta(fa*30.4375,unit='D')
    ax.plot(d0,s,'o-',color='C0',ms=3,lw=1.2)
    ax.plot(df_,sim['q50'],'-',color='C2',lw=1.8); ax.fill_between(df_,sim['q10'],sim['q90'],color='C2',alpha=0.18)
    ax.axhline(EOL,ls=':',color='red',lw=1)
    if we<=df_.max(): ax.axvline(we,ls='-.',color='green',lw=1.4)
    rul='%d mo'%k if len(cross) else '>%d mo'%MAXH
    eol_date=df_[-1].strftime('%Y-%m') if len(cross) else 'n/a'
    ax.set_title(f"{vin[-8:]} ({vin_model.get(vin,'?')})  {s[0]:.0f}->{s[-1]:.0f}%\nto 80%: {rul} ({eol_date}) | warr {wlab}",fontsize=8)
    ax.set_ylim(78,102); ax.grid(alpha=0.3); ax.tick_params(labelsize=7)
fig.suptitle('Oldest Mahindra vehicles — actual + custom forecast TO 80% EOL (warranty: 3yr Treo / 5yr Zor Grand or 120k km)',fontsize=12)
plt.tight_layout(rect=[0,0,1,0.96])
plt.show()
